In [16]:
"""
ANN模型（LayerNorm版）1D PDP完整分析程序
功能：1D部分依赖图分析（增强版梯度可视化）
作者：Claude
日期：2025-01-15
特点：1. 🔥 KQ范围过滤：只分析0 <= KQ <= 20.2的数据
      2. 🔥 三个置信区间（90%, 95%, 99%）
      3. 🔥 增强的梯度可视化（标注极值、零点、统计信息）
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
import torch.nn as nn
import warnings

warnings.filterwarnings("ignore")

# ====================== 配置部分 ======================
CONFIG = {
    # 数据和模型路径
    'data_file': r"D:\文成数据库\Nb-Si数据库3.4-成分-性能.xlsx",
    'model_dir': r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充",
    
    # 输出路径
    'output_base_dir': r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\ANN模型1D_PDP分析结果1.15最终版",
    
    # 模型的6个特征（从训练程序获得）
    'selected_features': [
        'Δx', 
        'ΔHmix', 
        'am', 
        'Ω', 
        'mean_C4 enthalpy melting',      # M-F4
        'mean_Electrophilicity Index'    # M-B3
    ],
    
    'target_col': 'KQ',
    
    # 🔥 KQ数据范围过滤
    'kq_min': 0,
    'kq_max': 20.2,
    
    # PDP分析参数
    'grid_resolution_1d': 200,      # 1D PDP的网格点数
    'percentiles': (0.01, 0.99),   # 特征值范围（百分位）
    'n_bootstrap': 1000,           # Bootstrap次数
    
    # 可视化参数
    'figure_dpi': 300,
    'figure_size': (12, 10),
}

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
sns.set_style("whitegrid")

# ====================== 1. 定义模型结构 ======================
class MyModel(nn.Module):
    """LayerNorm版ANN模型"""
    def __init__(self, input_dim=6):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 24)
        self.ln1 = nn.LayerNorm(24)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(24, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.ln1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# ====================== 2. 加载模型和数据 ======================
def load_model_and_data():
    """加载训练好的模型和数据"""
    print("="*80)
    print("ANN模型（LayerNorm版）1D PDP分析程序 - 增强版")
    print("="*80)
    print("\n步骤1: 加载模型和数据...")
    
    # 设置设备
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"  使用设备: {device}")
    
    # 加载模型配置
    config_path = os.path.join(CONFIG['model_dir'], 'model_config.pkl')
    with open(config_path, 'rb') as f:
        model_config = pickle.load(f)
    print(f"  ✓ 加载模型配置")
    
    # 加载标准化器
    scaler_path = os.path.join(CONFIG['model_dir'], 'scaler.pkl')
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    print(f"  ✓ 加载标准化器")
    
    # 加载训练/测试集索引
    indices_path = os.path.join(CONFIG['model_dir'], 'train_test_indices.pkl')
    with open(indices_path, 'rb') as f:
        indices_data = pickle.load(f)
    train_indices = indices_data['final_train_indices']
    test_indices = indices_data['final_test_indices']
    print(f"  ✓ 加载数据集划分")
    print(f"    - 训练集: {len(train_indices)} 样本")
    print(f"    - 测试集: {len(test_indices)} 样本")
    
    # 加载模型
    model_path = os.path.join(CONFIG['model_dir'], 'trained_model.pth')
    model = MyModel(len(CONFIG['selected_features'])).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"  ✓ 加载训练好的模型")
    print(f"    - 训练集R²: {model_config['final_train_r2']:.6f}")
    print(f"    - 测试集R²: {model_config['final_test_r2']:.6f}")
    
    # 加载原始数据
    df = pd.read_excel(CONFIG['data_file'])
    print(f"  ✓ 加载原始数据: {df.shape}")
    
    # 🔥 过滤KQ数据
    original_count = len(df)
    kq_values = df[CONFIG['target_col']].values
    valid_mask = (kq_values >= CONFIG['kq_min']) & (kq_values <= CONFIG['kq_max'])
    df_filtered = df[valid_mask].copy()
    filtered_count = len(df_filtered)
    removed_count = original_count - filtered_count
    
    print(f"\n  🔥 KQ数据过滤:")
    print(f"    - 原始样本数: {original_count}")
    print(f"    - 过滤后样本数: {filtered_count}")
    print(f"    - 移除样本数: {removed_count}")
    print(f"    - KQ范围限制: [{CONFIG['kq_min']}, {CONFIG['kq_max']}]")
    if removed_count > 0:
        removed_kq_values = kq_values[~valid_mask]
        print(f"    - 移除的KQ值范围: [{np.min(removed_kq_values):.2f}, {np.max(removed_kq_values):.2f}]")
    
    # 提取特征和目标
    X_original = df_filtered[CONFIG['selected_features']].values
    y_original = df_filtered[CONFIG['target_col']].values
    X_scaled = scaler.transform(X_original)
    
    print(f"\n数据信息（过滤后）:")
    print(f"  - 总样本数: {len(X_scaled)}")
    print(f"  - 特征数: {len(CONFIG['selected_features'])}")
    print(f"  - 目标变量范围: [{np.min(y_original):.2f}, {np.max(y_original):.2f}]")
    
    return {
        'model': model,
        'device': device,
        'scaler': scaler,
        'X_original': X_original,
        'X_scaled': X_scaled,
        'y': y_original,
        'model_config': model_config,
        'features': CONFIG['selected_features'],
        'filtered_info': {
            'original_count': original_count,
            'filtered_count': filtered_count,
            'removed_count': removed_count,
            'kq_range': [CONFIG['kq_min'], CONFIG['kq_max']]
        }
    }

# ====================== 3. PDP预测函数 ======================
def predict_pytorch(model, X_scaled, device):
    """使用PyTorch模型进行预测"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.from_numpy(X_scaled.astype(np.float32)).to(device)
        predictions = model(X_tensor).cpu().numpy().flatten()
    return predictions

# ====================== 4. 1D PDP分析 ======================
def calculate_1d_pdp(model_data, feature_idx, feature_name):
    """计算单个特征的1D PDP"""
    print(f"\n  处理特征 {feature_idx+1}/6: {feature_name}")
    
    model = model_data['model']
    device = model_data['device']
    scaler = model_data['scaler']
    X_original = model_data['X_original']
    X_scaled = model_data['X_scaled']
    
    # 获取特征范围
    feature_original = X_original[:, feature_idx]
    feature_min = np.percentile(feature_original, CONFIG['percentiles'][0] * 100)
    feature_max = np.percentile(feature_original, CONFIG['percentiles'][1] * 100)
    
    # 创建网格
    grid_original = np.linspace(feature_min, feature_max, CONFIG['grid_resolution_1d'])
    
    # 计算PDP
    pdp_values = []
    for grid_val in grid_original:
        X_modified_original = X_original.copy()
        X_modified_original[:, feature_idx] = grid_val
        X_modified_scaled = scaler.transform(X_modified_original)
        predictions = predict_pytorch(model, X_modified_scaled, device)
        pdp_values.append(np.mean(predictions))
    
    pdp_values = np.array(pdp_values)
    
    print(f"    - 特征范围: [{feature_min:.4f}, {feature_max:.4f}]")
    print(f"    - PDP范围: [{np.min(pdp_values):.4f}, {np.max(pdp_values):.4f}]")
    
    # Bootstrap置信区间
    print(f"    - Bootstrap（{CONFIG['n_bootstrap']}次）...")
    bootstrap_pdps = []
    n_samples = X_scaled.shape[0]
    
    for b in range(CONFIG['n_bootstrap']):
        boot_indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot_original = X_original[boot_indices]
        
        boot_pdp = []
        for grid_val in grid_original:
            X_modified_original = X_boot_original.copy()
            X_modified_original[:, feature_idx] = grid_val
            X_modified_scaled = scaler.transform(X_modified_original)
            predictions = predict_pytorch(model, X_modified_scaled, device)
            boot_pdp.append(np.mean(predictions))
        
        bootstrap_pdps.append(boot_pdp)
        
        if (b + 1) % 200 == 0:
            print(f"      进度: {b+1}/{CONFIG['n_bootstrap']}")
    
    bootstrap_pdps = np.array(bootstrap_pdps)
    
    # 计算三个置信区间
    ci_90_lower = np.percentile(bootstrap_pdps, 5, axis=0)
    ci_90_upper = np.percentile(bootstrap_pdps, 95, axis=0)
    ci_95_lower = np.percentile(bootstrap_pdps, 2.5, axis=0)
    ci_95_upper = np.percentile(bootstrap_pdps, 97.5, axis=0)
    ci_99_lower = np.percentile(bootstrap_pdps, 0.5, axis=0)
    ci_99_upper = np.percentile(bootstrap_pdps, 99.5, axis=0)
    
    return {
        'feature_name': feature_name,
        'grid_values': grid_original,
        'pdp_values': pdp_values,
        'ci_90_lower': ci_90_lower,
        'ci_90_upper': ci_90_upper,
        'ci_95_lower': ci_95_lower,
        'ci_95_upper': ci_95_upper,
        'ci_99_lower': ci_99_lower,
        'ci_99_upper': ci_99_upper,
        'feature_data': feature_original
    }

def visualize_1d_pdp(pdp_result, output_dir):
    """🔥 增强的1D PDP可视化（含梯度增强）"""
    feature_name = pdp_result['feature_name']
    grid_values = pdp_result['grid_values']
    pdp_values = pdp_result['pdp_values']
    ci_90_lower = pdp_result['ci_90_lower']
    ci_90_upper = pdp_result['ci_90_upper']
    ci_95_lower = pdp_result['ci_95_lower']
    ci_95_upper = pdp_result['ci_95_upper']
    ci_99_lower = pdp_result['ci_99_lower']
    ci_99_upper = pdp_result['ci_99_upper']
    feature_data = pdp_result['feature_data']
    
    safe_name = feature_name.replace(' ', '_').replace('/', '_')
    
    # 创建图形
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    # ========== 子图1: PDP + 置信区间 + 特征分布 ==========
    ax1 = axes[0]
    
    # 三个置信区间
    ax1.fill_between(grid_values, ci_99_lower, ci_99_upper, 
                     alpha=0.15, color='lightblue', label='99% CI')
    ax1.fill_between(grid_values, ci_95_lower, ci_95_upper, 
                     alpha=0.25, color='lightblue', label='95% CI')
    ax1.fill_between(grid_values, ci_90_lower, ci_90_upper, 
                     alpha=0.35, color='lightblue', label='90% CI')
    
    # PDP曲线
    ax1.plot(grid_values, pdp_values, 'b-', linewidth=3, label='PDP')
    
    # 特征分布（右y轴）
    ax1_twin = ax1.twinx()
    ax1_twin.hist(feature_data, bins=50, alpha=0.3, color='gray', density=True)
    ax1_twin.set_ylabel('Density', fontsize=12, color='gray')
    ax1_twin.tick_params(axis='y', labelcolor='gray')
    
    ax1.set_xlabel(feature_name, fontsize=13, fontweight='bold')
    ax1.set_ylabel('PDP (KQ)', fontsize=13, fontweight='bold')
    ax1.set_title(f'Partial Dependence Plot of {feature_name}', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # ========== 🔥 子图2: 增强的梯度图 ==========
    ax2 = axes[1]
    
    if len(grid_values) > 1:
        gradient = np.gradient(pdp_values, grid_values)
        
        # 1. 区域背景色
        y_lim = ax2.get_ylim()
        ax2.axhspan(0, 5, alpha=0.05, color='green', zorder=0)  # 正梯度区
        ax2.axhspan(-5, 0, alpha=0.05, color='red', zorder=0)   # 负梯度区
        
        # 2. 基础梯度曲线
        ax2.plot(grid_values, gradient, 'r-', linewidth=2, label='∂PDP/∂' + feature_name, zorder=3)
        ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5, zorder=2)
        
        # 3. 🔥 标注极值点
        max_idx = np.argmax(gradient)
        min_idx = np.argmin(gradient)
        
        # 最大值标注
        ax2.scatter(grid_values[max_idx], gradient[max_idx], 
                   s=100, c='darkred', marker='o', zorder=5, 
                   edgecolors='black', linewidth=1.5)
        ax2.annotate(f'{gradient[max_idx]:.2f}',
                    xy=(grid_values[max_idx], gradient[max_idx]),
                    xytext=(grid_values[max_idx], gradient[max_idx] + 0.15),
                    fontsize=10, fontweight='bold', color='darkred',
                    ha='center',
                    bbox=dict(boxstyle='round,pad=0.4', 
                            facecolor='white', edgecolor='darkred', linewidth=1.5),
                    arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))
        
        # 最小值标注
        ax2.scatter(grid_values[min_idx], gradient[min_idx],
                   s=100, c='darkblue', marker='o', zorder=5,
                   edgecolors='black', linewidth=1.5)
        ax2.annotate(f'{gradient[min_idx]:.2f}',
                    xy=(grid_values[min_idx], gradient[min_idx]),
                    xytext=(grid_values[min_idx], gradient[min_idx] - 0.15),
                    fontsize=10, fontweight='bold', color='darkblue',
                    ha='center',
                    bbox=dict(boxstyle='round,pad=0.4', 
                            facecolor='white', edgecolor='darkblue', linewidth=1.5),
                    arrowprops=dict(arrowstyle='->', color='darkblue', lw=1.5))
        
        # 4. 🔥 标注零点
        zero_crossings = np.where(np.diff(np.sign(gradient)))[0]
        for zc in zero_crossings:
            ax2.axvline(x=grid_values[zc], color='gray', 
                       linestyle=':', linewidth=1.5, alpha=0.6, zorder=1)
            ax2.text(grid_values[zc], ax2.get_ylim()[0] * 0.9, 
                    f'{grid_values[zc]:.1f}',
                    rotation=90, fontsize=8, color='gray',
                    verticalalignment='bottom', ha='right')
        
        # 5. 🔥 统计信息框
        stats_text = (
            f'Max: {gradient[max_idx]:.3f}\n'
            f'@ {feature_name}={grid_values[max_idx]:.2f}\n'
            f'\n'
            f'Min: {gradient[min_idx]:.3f}\n'
            f'@ {feature_name}={grid_values[min_idx]:.2f}\n'
            f'\n'
            f'Mean |∂PDP|: {np.mean(np.abs(gradient)):.3f}'
        )
        ax2.text(0.98, 0.97, stats_text,
                transform=ax2.transAxes,
                verticalalignment='top',
                horizontalalignment='right',
                fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', 
                         alpha=0.8, edgecolor='black', linewidth=1.5))
        
        ax2.set_xlabel(feature_name, fontsize=12, fontweight='bold')
        ax2.set_ylabel('PDP Gradient', fontsize=12, fontweight='bold')
        ax2.set_title('Feature Effect Gradient', fontsize=12, fontweight='bold')
        ax2.legend(loc='upper left', fontsize=9)
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # 保存图片
    img_path = os.path.join(output_dir, f'1D_PDP_{safe_name}.png')
    plt.savefig(img_path, dpi=CONFIG['figure_dpi'], bbox_inches='tight')
    plt.close()
    
    print(f"      ✓ 保存图片: 1D_PDP_{safe_name}.png")
    
    # 保存数据到Excel
    excel_data = pd.DataFrame({
        'feature_value': grid_values,
        'pdp_value': pdp_values,
        'ci_90_lower': ci_90_lower,
        'ci_90_upper': ci_90_upper,
        'ci_95_lower': ci_95_lower,
        'ci_95_upper': ci_95_upper,
        'ci_99_lower': ci_99_lower,
        'ci_99_upper': ci_99_upper
    })
    
    excel_path = os.path.join(output_dir, f'1D_PDP_Data_{safe_name}.xlsx')
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        excel_data.to_excel(writer, sheet_name='PDP数据', index=False)
        
        # 特征分布
        feature_df = pd.DataFrame({'feature_value': feature_data})
        feature_df.to_excel(writer, sheet_name='特征分布', index=False)
        
        # 梯度数据
        if len(grid_values) > 1:
            gradient = np.gradient(pdp_values, grid_values)
            gradient_df = pd.DataFrame({
                'feature_value': grid_values,
                'gradient': gradient
            })
            gradient_df.to_excel(writer, sheet_name='梯度数据', index=False)
    
    print(f"      ✓ 保存数据: 1D_PDP_Data_{safe_name}.xlsx")

# ====================== 5. 主程序 ======================
def main():
    """主程序"""
    print("\n开始ANN模型1D PDP分析（增强版）...")
    
    # 创建输出目录
    output_dir = CONFIG['output_base_dir']
    os.makedirs(output_dir, exist_ok=True)
    print(f"输出目录: {output_dir}")
    
    # 加载模型和数据
    model_data = load_model_and_data()
    
    # ========== 1D PDP分析 ==========
    print("\n" + "="*80)
    print("步骤2: 1D PDP分析（所有6个特征）")
    print("="*80)
    
    n_features = len(CONFIG['selected_features'])
    pdp_1d_results = []
    
    for i in range(n_features):
        feature_name = CONFIG['selected_features'][i]
        pdp_result = calculate_1d_pdp(model_data, i, feature_name)
        visualize_1d_pdp(pdp_result, output_dir)
        pdp_1d_results.append(pdp_result)
    
    print(f"\n✓ 完成1D PDP分析：{n_features}个特征")
    
    # ========== 生成报告 ==========
    print("\n" + "="*80)
    print("步骤3: 生成分析报告")
    print("="*80)
    
    report_path = os.path.join(output_dir, 'PDP_Analysis_Report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("ANN模型 1D PDP分析报告（增强版）\n")
        f.write("="*80 + "\n\n")
        
        f.write("1. 模型信息\n")
        f.write("-"*80 + "\n")
        f.write(f"模型类型: LayerNorm ANN\n")
        f.write(f"输入特征数: {n_features}\n")
        f.write(f"训练集R²: {model_data['model_config']['final_train_r2']:.6f}\n")
        f.write(f"测试集R²: {model_data['model_config']['final_test_r2']:.6f}\n\n")
        
        f.write("2. 数据过滤信息\n")
        f.write("-"*80 + "\n")
        filtered_info = model_data['filtered_info']
        f.write(f"KQ范围限制: [{filtered_info['kq_range'][0]}, {filtered_info['kq_range'][1]}]\n")
        f.write(f"原始样本数: {filtered_info['original_count']}\n")
        f.write(f"过滤后样本数: {filtered_info['filtered_count']}\n")
        f.write(f"移除样本数: {filtered_info['removed_count']}\n\n")
        
        f.write("3. 分析特征\n")
        f.write("-"*80 + "\n")
        for i, feat in enumerate(CONFIG['selected_features'], 1):
            f.write(f"  {i}. {feat}\n")
        f.write("\n")
        
        f.write("4. 分析配置\n")
        f.write("-"*80 + "\n")
        f.write(f"网格分辨率: {CONFIG['grid_resolution_1d']} 点\n")
        f.write(f"Bootstrap次数: {CONFIG['n_bootstrap']}\n")
        f.write(f"置信区间: 90%, 95%, 99%\n")
        f.write(f"特征范围: {CONFIG['percentiles'][0]*100}%-{CONFIG['percentiles'][1]*100}%分位\n\n")
        
        f.write("5. 输出文件\n")
        f.write("-"*80 + "\n")
        f.write(f"每个特征生成:\n")
        f.write(f"  - 1个PNG图片（PDP + 增强梯度图）\n")
        f.write(f"  - 1个Excel文件（含PDP数据、特征分布、梯度数据）\n")
        f.write(f"\n总计: {n_features} × 2 = {n_features*2} 个文件\n\n")
        
        f.write("6. 增强功能\n")
        f.write("-"*80 + "\n")
        f.write("✓ KQ范围过滤\n")
        f.write("✓ 三个置信区间（90%, 95%, 99%）\n")
        f.write("✓ 梯度极值点标注\n")
        f.write("✓ 零点（转折点）标注\n")
        f.write("✓ 正负效应区域背景色\n")
        f.write("✓ 统计信息文本框\n")
        f.write("✓ 完整数据导出到Excel\n\n")
        
        f.write("="*80 + "\n")
        f.write("分析完成时间: 2025-01-15\n")
        f.write("="*80 + "\n")
    
    print(f"✓ 分析报告已保存: {report_path}")
    
    # ========== 完成 ==========
    print("\n" + "="*80)
    print("ANN模型1D PDP分析完成（增强版）！")
    print("="*80)
    print(f"\n所有结果已保存到: {output_dir}")
    print(f"\n生成的文件:")
    print(f"  • {n_features} 个特征分析图片（PNG）")
    print(f"  • {n_features} 个数据文件（Excel）")
    print(f"  • 1 个综合报告（TXT）")
    print(f"  • 总计: {n_features*2 + 1} 个文件")
    print(f"\n增强功能:")
    print(f"  ✓ KQ范围过滤: [{CONFIG['kq_min']}, {CONFIG['kq_max']}]")
    print(f"  ✓ 移除 {model_data['filtered_info']['removed_count']} 个超出范围样本")
    print(f"  ✓ 三个置信区间（90%, 95%, 99%）")
    print(f"  ✓ 梯度图增强（极值标注、零点标注、统计信息）")
    print(f"  ✓ 完整数据保存（PDP + 特征分布 + 梯度）")
    
if __name__ == "__main__":
    main()


开始ANN模型1D PDP分析（增强版）...
输出目录: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\ANN模型1D_PDP分析结果1.15最终版
ANN模型（LayerNorm版）1D PDP分析程序 - 增强版

步骤1: 加载模型和数据...
  使用设备: cpu
  ✓ 加载模型配置
  ✓ 加载标准化器
  ✓ 加载数据集划分
    - 训练集: 176 样本
    - 测试集: 45 样本
  ✓ 加载训练好的模型
    - 训练集R²: 0.954897
    - 测试集R²: 0.967866
  ✓ 加载原始数据: (221, 104)

  🔥 KQ数据过滤:
    - 原始样本数: 221
    - 过滤后样本数: 210
    - 移除样本数: 11
    - KQ范围限制: [0, 20.2]
    - 移除的KQ值范围: [22.58, 62.50]

数据信息（过滤后）:
  - 总样本数: 210
  - 特征数: 6
  - 目标变量范围: [3.40, 20.20]

步骤2: 1D PDP分析（所有6个特征）

  处理特征 1/6: Δx
    - 特征范围: [5.6040, 29.8222]
    - PDP范围: [4.5584, 11.2243]
    - Bootstrap（1000次）...
      进度: 200/1000
      进度: 400/1000
      进度: 600/1000
      进度: 800/1000
      进度: 1000/1000
      ✓ 保存图片: 1D_PDP_Δx.png
      ✓ 保存数据: 1D_PDP_Data_Δx.xlsx

  处理特征 2/6: ΔHmix
    - 特征范围: [-36.0005, -4.4344]
    - PDP范围: [11.1108, 16.5692]
    - Bootstrap（1000次）...
      进度: 200/1000
      进度: 400/1000
      进度: 600/1000
      进度: 800/1000
      进度: 1000/1000
      ✓